# NHCS Scenario 1 — A million visits: who comes to the emergency department, and why

_Describing the largest dataset of this MIP. ED dataset only. Level: first analysis on this pathology, no statistics background needed. About 60 minutes._

**Before you start.** The NHCS pathology holds two datasets from about 200 US hospitals, 2020 to
2022: 1,368,648 emergency department visit records and 488,636 inpatient discharge records, each a
5% sample of all encounters of the participating hospitals. **A row is an encounter record, not a
patient**, and the two datasets cannot be linked: each is an independent sample, the identifiers
were renumbered, and NCHS merged the ED outcome "admitted" into "other". The MIP does not use the
survey weights: every figure describes *the sampled encounters*, not US hospital care. Codes are
NCHS codes: Sex 1 = Male, 2 = Female; NCHS flags 1 = Yes, 2 = No; curated flags 0 = No, 1 = Yes;
age is top-coded at 86 and the length of stay at 15 days.

**How this notebook works.** Every analysis is a *federated* call: the code selects a dataset and
variables, defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a
model). No record ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect. These are the largest datasets of this MIP: the notebook prints how
long the regressions take.

**Learning goals.** Describe a dataset of 1.4 million records with a handful of calls; read an age
histogram; use grouped diagnoses and condition flags instead of raw codes; tell a flag that is
absent from a flag that was never delivered; cross-tabulate a condition by age group; and read how
visits end when the most important outcome, admission, has been masked.

**Clinical background.** An emergency department sees everything: injuries, chest and abdominal
pain, infections, mental health crises, the complications of chronic disease, and the simply
worried. Age shapes the mix. The NHCS ED file records, for a 5% sample of all visits to about 200
US hospitals in 2020 to 2022, when the visit was, who the patient was, up to 30 diagnoses, and how
the visit ended. Diagnoses are summarised in chapters, in a 40-group first-listed diagnosis and in
condition flags, because raw codes have thousands of values.

## 1. Connect to the MIP and open the NHCS pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("NHCS")        # add version="ED_Inpatient_v1" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
def dataset(code, label):
    # by the code, which is how the platform labels its datasets (seen 2026-10-12), else by the long label
    try:
        return dm.datasets[code]
    except (KeyError, LookupError):
        return dm.datasets[label]

ed = dataset("nhcs_ed", "NHCS emergency department visits, USA, 2020-2022")
ip = dataset("nhcs_ip", "NHCS inpatient discharges, USA, 2020-2022")
display(dm)
display(dm.datasets.to_frame())

In [ ]:
# The variable tree is the anatomy of one encounter record: Encounter, Patient, Discharge,
# Diagnoses (with the curated condition flags and the NCHS CCSR flags), and the branch
# Inpatient stay (empty for ED visits): length of stay and procedures.
dm.tree()

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the selected datasets
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False, datasets=None):
    """A Pipeline on the datasets of this notebook (DATASETS, or the ones given) with the given
    variables and cohort filter. Tests and models need drop_missing=True: the platform requires an
    explicit missing-values step and analyses complete cases only. describe() keeps it off so it
    can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=datasets or DATASETS, variables=variables),
                        filters=filters, handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def rows_for(result, variable, dataset=POOLED):
    """The describe row of one variable: the pooled row by default, or the row of one Dataset."""
    keys = (POOLED,) if dataset == POOLED else (dataset._code, dataset.label)
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") in keys]


def numeric_summary(result, variables, dataset=POOLED):
    """n, missing, mean, SD, quartiles and range of numerical variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v, dataset):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "q1": d.get("q1"), "median": d.get("q2"),
                         "q3": d.get("q3"), "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable, dataset=POOLED):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable, dataset):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan, "missing": (r.get("data") or {}).get("num_na")})
    return pd.DataFrame(rows)


def level_percent(result, variable, levels=("Yes",), dataset=POOLED):
    """Percentage of the given levels (labels) of a categorical variable in a describe result."""
    k = category_counts(result, variable, dataset)
    return float(k.loc[k["level"].isin(levels), "percent"].sum()) if len(k) else np.nan


def by_level(numeric, group, levels=None, base_filter=None, datasets=None):
    """n, mean, median and quartiles of a numerical variable in each level of a categorical one
    (one describe per level). levels: the codes to use, by default every code of the variable."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({group.label: lab.get(c, c), "n": d.get("num_dtps"), "mean": d.get("mean"),
                         "q1": d.get("q1"), "median": d.get("q2"), "q3": d.get("q3")})
    return pd.DataFrame(rows)


def band_means(numeric, bands, base_filter=None, datasets=None):
    """n, mean and median of a numerical variable in bands given as {label: filter}."""
    rows = []
    for label, f in bands.items():
        if base_filter is not None:
            f = f & base_filter
        res = analysis([numeric], f, datasets=datasets).describe()
        for r in rows_for(res, numeric):
            d = r.get("data") or {}
            rows.append({"band": label, "n": d.get("num_dtps"), "mean": d.get("mean"), "median": d.get("q2")})
    return pd.DataFrame(rows)


def crosstab(exposure, outcome, base_filter=None, event="1", levels=None, event_label=None, datasets=None):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort.
    `event` is the outcome code counted in the percentage column, or a list of codes."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    events = [event] if isinstance(event, str) else list(event)
    pct_col = "% " + (event_label or lab_y.get(events[0], events[0]))
    rows = []
    for c in levels or lab_x:
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f, datasets=datasets).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: lab_x.get(c, c), "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row[pct_col] = 100 * sum(counts.get(e, 0) for e in events) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if levels:
        f = f & F(exposure).isin(list(levels))
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True, datasets=datasets).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def rates_by(group, outcomes, levels=None, base_filter=None, event="1", datasets=None):
    """% of records with each outcome (code `event`) in every level of `group`: one describe per
    level for the counts of all outcomes, one Chi-squared Test per outcome (in .attrs)."""
    lab = labels_of(group)
    rows = []
    for c in levels or lab:
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(outcomes, f, datasets=datasets).describe()
        row = {group.label: lab.get(c, c)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get(event, 0) / n if n else np.nan
        rows.append(row)
    table = pd.DataFrame(rows)
    table.attrs["chi2"], table.attrs["p"] = {}, {}
    for y in outcomes:
        f = F(group).is_not_null() & F(y).is_not_null()
        if levels:
            f = f & F(group).isin(list(levels))
        if base_filter is not None:
            f = f & base_filter
        chi = analysis([group, y], f, drop_missing=True, datasets=datasets).chi_square_test(x=group, y=y)
        table.attrs["chi2"][y.label] = chi.raw.get("chi2")
        table.attrs["p"][y.label] = chi.raw.get("p_value")
    return table


def month_label(i):
    """Month index 1..36 -> '2020-01' .. '2022-12'."""
    return f"{2020 + (i - 1) // 12}-{(i - 1) % 12 + 1:02d}"


def by_month(outcomes, variable, months=range(1, 37), datasets=None):
    """% of records with each outcome (code 1) per month of the time axis: one describe per month."""
    rows = []
    for m in months:
        res = analysis(outcomes, F(variable) == m, datasets=datasets).describe()
        row = {"month": m, "label": month_label(m)}
        for y in outcomes:
            counts = {}
            for r in rows_for(res, y):
                counts = (r.get("data") or {}).get("counts") or {}
            n = sum(counts.values())
            row.setdefault("n", n)
            row[y.label] = 100 * counts.get("1", 0) / n if n else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None, rotate=0):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: PCT}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:,.0f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(7, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=rotate)
    plt.tight_layout()
    plt.show()


def anova_summary(result):
    """One-way ANOVA result -> F, p, n, a table of the group means and SDs, and the Tukey comparisons."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    table, ci = raw.get("anova_table", {}), raw.get("ci_info", {})
    groups = pd.DataFrame({"mean": pd.Series(ci.get("means", {})), "sd": pd.Series(ci.get("sample_stds", {}))})
    return table.get("f_stat"), table.get("p_value"), table.get("n_obs"), groups, pd.DataFrame(raw.get("tuckey_test", []))


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def n_fmt(value):
    # a count with thousands separators, or n/a when the platform did not return it
    return f"{value:,}" if isinstance(value, (int, float)) else "n/a"


def ttest_from_describe(result_a, result_b, variable):
    """The pooled-variance t statistic of two groups from their describe results (n, mean, SD):
    all a t-test needs, for groups the platform's t-test cannot form (several codes pooled)."""
    a = numeric_summary(result_a, [variable]).iloc[0]
    b = numeric_summary(result_b, [variable]).iloc[0]
    n1, n2 = a["n"], b["n"]
    sp2 = ((n1 - 1) * a["sd"] ** 2 + (n2 - 1) * b["sd"] ** 2) / (n1 + n2 - 2)
    t = (a["mean"] - b["mean"]) / math.sqrt(sp2 * (1 / n1 + 1 / n2))
    return {"mean_a": a["mean"], "mean_b": b["mean"], "diff": a["mean"] - b["mean"], "t": t, "n": n1 + n2}


def term_label(term, variables):
    """Model term -> readable label: 'SEX[2]' -> 'Sex = Female', 'AGE' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(lambda t: term_label(t, variables))
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def linear_table(result, variables):
    """Linear regression result -> one row per term with coefficient (units of the outcome), 95% CI and p."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return pd.DataFrame({"term": [term_label(t, variables) for t in raw.get("indep_vars", [])],
                         "coefficient": raw.get("coefficients"), "ci_low": raw.get("lower_ci"),
                         "ci_high": raw.get("upper_ci"), "p": raw.get("pvalues")})


def show_coefficients(table, title, unit="days"):
    """Table and dot chart of linear-regression coefficients (the intercept left out of the chart)."""
    display(table.style.format({"coefficient": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table[table["term"] != "Intercept"].iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["coefficient"], range(len(t)),
                xerr=[t["coefficient"] - t["ci_low"], t["ci_high"] - t["coefficient"]], fmt="o", capsize=4)
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel(f"{unit} added, for the same values of the other predictors")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The dataset and the variables of this scenario

| Code | Label in the MIP | Tree location (under NHCS) | Role | Values |
|---|---|---|---|---|
| `YEAR` | Survey year | Encounter | volume by year | 2020 2020; 2021 2021; 2022 2022 |
| `SEX` | Sex | Patient | who comes | 1 Male; 2 Female |
| `AGE` | Age | Patient | who comes; histogram | integer, 0-86 years |
| `age_group` | Age group | Patient | who comes; grouping of Steps 4 and 5; curation-derived | 1 Under 1 year; 2 1-17 years; 3 18-44 years; 4 45-64 years; 5 65-74 years; 6 75-84 years; 7 85 years and over |
| `dx1_chapter` | First-listed diagnosis, ICD-10 chapter | Diagnoses | why they come; curation-derived | 22 codes, listed in the tree |
| `dx1_group` | First-listed diagnosis, group | Diagnoses | why they come; curation-derived | 40 codes, listed in the tree |
| `n_dx` | Number of diagnoses | Diagnoses | number of diagnoses (extension); curation-derived | integer, 0-30 |
| `cond_injury` | Injury diagnosis | Diagnoses / Condition flags (curated) | curated flag; curation-derived | 0 No; 1 Yes |
| `cond_hypertension` | Hypertension | Diagnoses / Condition flags (curated) | curated flag; curation-derived | 0 No; 1 Yes |
| `cond_diabetes` | Diabetes mellitus | Diagnoses / Condition flags (curated) | curated flag; curation-derived | 0 No; 1 Yes |
| `cond_tobacco` | Tobacco use or nicotine dependence | Diagnoses / Condition flags (curated) | curated flag; curation-derived | 0 No; 1 Yes |
| `cond_asthma` | Asthma | Diagnoses / Condition flags (curated) | curated flag; by age; curation-derived | 0 No; 1 Yes |
| `cond_pregnancy` | Pregnancy-related diagnosis | Diagnoses / Condition flags (curated) | curated flag; extension; curation-derived | 0 No; 1 Yes |
| `cond_alcohol` | Alcohol-related disorder | Diagnoses / Condition flags (curated) | curated flag; by age; curation-derived | 0 No; 1 Yes |
| `covid_dx` | COVID-19, any position | Diagnoses / Condition flags (curated) | curated flag, all years; curation-derived | 0 No; 1 Yes |
| `CCSR_CIR012` | CCSR CIR012: Nonspecific chest pain (ED) | Diagnoses / CCSR flags (NCHS) | NCHS flag: chest pain; by age | 1 Yes; 2 No |
| `CCSR_SYM006` | CCSR SYM006: Abdominal pain and other digestive/abdomen signs and symptoms (ED) | Diagnoses / CCSR flags (NCHS) | NCHS flag: abdominal pain | 1 Yes; 2 No |
| `CCSR_SYM013` | CCSR SYM013: Respiratory signs and symptoms (ED) | Diagnoses / CCSR flags (NCHS) | NCHS flag: respiratory symptoms | 1 Yes; 2 No |
| `CCSR_INF012` | CCSR INF012: Coronavirus disease 2019 (COVID-19) (ED 2021-2022) | Diagnoses / CCSR flags (NCHS) | NCHS flag delivered for 2021-2022 only | 1 Yes; 2 No |
| `CCSR_EXT020` | CCSR EXT020: External cause codes: intent of injury, accidental/unintentional (ED 2020) | Diagnoses / CCSR flags (NCHS) | NCHS flag delivered for 2020 only | 1 Yes; 2 No |
| `discharge_group` | Discharge group | Discharge | how the visit ended; curation-derived | 1 Home, routine; 2 Home with home health care; 3 Transfer to another facility or hospice; 4 Left against medical advice; 5 Died; 6 Other (for ED visits includes admission) |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `AGE`) is what the csv
column is called; you will see it in some result payloads. The first line below chooses the
dataset(s) every call of this notebook uses.

In [ ]:
DATASETS = [ed]                          # every call of this notebook runs on the ED visits

year       = dm.variables["Survey year"]
sex        = dm.variables["Sex"]
age        = dm.variables["Age"]
age_group  = dm.variables["Age group"]
chapter    = dm.variables["First-listed diagnosis, ICD-10 chapter"]
group      = dm.variables["First-listed diagnosis, group"]
n_dx       = dm.variables["Number of diagnoses"]
injury     = dm.variables["Injury diagnosis"]
hypertension = dm.variables["Hypertension"]
diabetes   = dm.variables["Diabetes mellitus"]
tobacco    = dm.variables["Tobacco use or nicotine dependence"]
asthma     = dm.variables["Asthma"]
pregnancy  = dm.variables["Pregnancy-related diagnosis"]
alcohol    = dm.variables["Alcohol-related disorder"]
covid      = dm.variables["COVID-19, any position"]
chest_pain = dm.variables["CCSR CIR012: Nonspecific chest pain (ED)"]
[abdominal_pain] = dm.variables.search("CCSR SYM006")          # the NCHS flags are found by their code
[resp_symptoms]  = dm.variables.search("CCSR SYM013")
[ccsr_covid]     = dm.variables.search("CCSR INF012")
[ext_cause]      = dm.variables.search("CCSR EXT020")
outcome    = dm.variables["Discharge group"]
display(mip.to_frame([year, sex, age, age_group, chapter, group, n_dx, injury, hypertension, diabetes, tobacco,
                      asthma, pregnancy, alcohol, covid, chest_pain, abdominal_pain, resp_symptoms, ccsr_covid, ext_cause, outcome]))

## Step 1. Who comes

One `describe()` of the year, sex, age and age group of 1.4 million visits, then a histogram of
age.

*Expected:* 1,368,648 visits: 388,753 in 2020, 482,126 in 2021, 497,769 in 2022. Women 53.9% of
the 1,362,086 visits with a recorded sex (6,562 unknown). Age 41.7 years on average (SD 23.9),
median 40, from 0 to the top code 86. Age groups: under 1 year 2.2%, 1 to 17 years 14.7%, 18 to
44 years 38.5%, 45 to 64 years 24.2%, 65 to 74 years 9.9%, 75 to 84 years 6.8%, 85 and over 3.8%.
The histogram has two peaks, the youngest children and the late twenties, then declines slowly;
the last bar gathers everyone from 82 to the top code.

In [ ]:
who = analysis([year, sex, age, age_group]).describe()
display(numeric_summary(who, [age]).style.format({c: "{:.1f}" for c in ("mean", "sd")}).hide(axis="index"))
counts = pd.concat([category_counts(who, v) for v in (year, sex, age_group)], ignore_index=True)
display(counts.style.format({"percent": PCT}).hide(axis="index"))
print(f"{int(counts.loc[counts['variable'] == year.label, 'count'].sum()):,} visits")

h = analysis([age], drop_missing=True).histogram(variable=age, bins=20).raw["histogram"][0]
edges, bars = h["bins"], [c or 0 for c in h["counts"]]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(edges[:-1], bars, width=np.diff(edges), align="edge", color="#4C72B0")
ax.set_xlabel("age (years, top code 86)"); ax.set_ylabel("visits"); ax.set_title("Age of 1.36 million sampled ED visits")
plt.tight_layout()
plt.show()

## Step 2. Why they come

The first-listed diagnosis in ICD-10 chapters and in the 40 curated groups.

*Expected, chapters:* symptoms, signs and abnormal findings 23.4%; injury and poisoning 16.6%;
respiratory 7.9%; musculoskeletal 7.4%; digestive 6.0%; genitourinary 5.1%; mental and
behavioural 5.0%; circulatory 4.7%; 45,294 visits (3.3%) carry no diagnosis at all.

*Expected, groups (the first twelve):* other symptoms and signs 14.2%, other injury including
burns 11.6%, musculoskeletal 7.4%, digestive diseases 6.0%, abdominal and pelvic pain 4.4%, chest
pain 4.3%, acute upper respiratory infection and influenza 4.1%, other nervous system, eye and ear
4.0%, health status and contact with services 3.5%, other genitourinary 3.1%, COVID-19 3.0%,
fracture 2.8%.

In [ ]:
why = analysis([chapter, group]).describe()
chapters = category_counts(why, chapter).sort_values("percent", ascending=False)
display(chapters.head(10)[["level", "count", "percent"]].style.format({"percent": PCT}).hide(axis="index"))
print(f"Visits without any diagnosis: {int(chapters['missing'].iloc[0]):,}")
groups = category_counts(why, group).sort_values("percent", ascending=False)
display(groups.head(12)[["level", "count", "percent"]].style.format({"percent": PCT}).hide(axis="index"))
ax = groups.head(12).set_index("level")["percent"].iloc[::-1].plot(kind="barh", color="#4C72B0", figsize=(8, 4.5))
ax.set_xlabel("% of visits"); ax.set_ylabel(""); ax.set_title("First-listed diagnosis, the twelve most frequent groups")
plt.tight_layout()
plt.show()

## Step 3. The flags, and a flag that was never delivered

The curated condition flags exist in every record (0 or 1). The NCHS CCSR flags were delivered
for the 20 most frequent categories of each yearly file, so a flag can be missing for a whole
year; the `describe()` result shows the missing values.

*Expected, curated flags:* injury 16.8%, hypertension 19.5%, tobacco 10.5%, diabetes 8.9%,
asthma 5.4%, COVID-19 in any position 4.1%, pregnancy 3.0%, alcohol 2.6%, never missing.

*Expected, NCHS flags:* chest pain 7.2%, abdominal pain 11.3%, respiratory symptoms 9.0% of the
1,323,365 visits that carry the flag (the 45,283 visits without any diagnosis have no flags). The
COVID-19 flag is missing for all 388,753 visits of 2020 and gives 4.8% of the visits of 2021 and
2022; the external-cause flag exists for 2020 only and is missing for the 979,895 visits of 2021
and 2022.

In [ ]:
curated = [injury, hypertension, tobacco, diabetes, asthma, covid, pregnancy, alcohol]
res = analysis(curated).describe()
display(pd.DataFrame({"% of visits": {v.label: level_percent(res, v) for v in curated},
                      "missing": {v.label: int(category_counts(res, v)["missing"].iloc[0]) for v in curated}}).style.format({"% of visits": PCT}))

nchs = [chest_pain, abdominal_pain, resp_symptoms, ccsr_covid, ext_cause]
res = analysis(nchs).describe()
flags = pd.DataFrame({"% yes, of the visits with the flag": {v.label: level_percent(res, v) for v in nchs},
                      "visits with the flag": {v.label: int(category_counts(res, v)["count"].sum()) for v in nchs},
                      "missing": {v.label: int(category_counts(res, v)["missing"].iloc[0]) for v in nchs}})
display(flags.style.format({"% yes, of the visits with the flag": PCT}))

# The same two NCHS flags, year by year: delivered, or not
rows = []
for y in ("2020", "2021", "2022"):
    r = analysis([ccsr_covid, ext_cause], F(year) == y).describe()
    for v in (ccsr_covid, ext_cause):
        k = category_counts(r, v)
        rows.append({"year": y, "flag": v.label[:30], "with the flag": int(k["count"].sum()) if len(k) else 0,
                     "missing": int(k["missing"].iloc[0]) if len(k) else None, "% yes": level_percent(r, v) if len(k) else np.nan})
display(pd.DataFrame(rows).style.format({"% yes": PCT}).hide(axis="index"))

*What it means:* a missing NCHS flag means "not delivered", not "not present". The names of
these variables carry the years; the curated flags, computed from the diagnosis codes in every
file, have no such gaps and are the ones to use across years.

## Step 4. Conditions by age

One `describe()` per age group of four flags, then one chi-square per flag (the `rates_by`
helper).

*Expected (percentage of the visits of the age group):* injury 7.1, 23.1, 16.1, 14.4, 14.6, 16.5,
21.7 from the youngest to the oldest group; chest pain 0.1, 1.9, 7.6, 10.4, 8.8, 7.0, 5.4; asthma
0.7, 6.9, 6.5, 5.3, 3.7, 3.0, 2.0; alcohol-related 0.0, 0.2, 3.0, 4.6, 2.2, 0.7, 0.3. All
p < 0.001 (chi-square 10,624, 16,126, 6,270 and 13,592). Injuries follow a U, chest pain peaks
in middle age, asthma in youth, alcohol at 45 to 64; with 1.4 million visits every p-value is
tiny, and the shapes of the curves carry the information.

In [ ]:
by_age = rates_by(age_group, [injury, chest_pain, asthma, alcohol]).set_index("Age group")
display(by_age.style.format({c: PCT for c in by_age.columns if c != "n"}))
for label, chi2 in by_age.attrs["chi2"].items():
    print(f"{label}: chi2 = {chi2:,.0f}, p = {fmt_p(by_age.attrs['p'][label])}")
ax = by_age.drop(columns="n").rename(columns=lambda c: c.split(":")[-1].strip()).plot(marker="o", figsize=(8, 3.8))
ax.set_ylabel("% of the visits of the age group"); ax.set_xlabel(""); ax.set_title("Four conditions by age group")
ax.legend(frameon=False, fontsize=8)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Step 5. How the visit ended

The discharge group on all visits, then by age group.

*Expected:* of the 1,302,296 visits with a known outcome, 87.3% went home (routine), 1.6% home
with home health care, 2.1% were transferred to another facility or hospice, 3.1% left against
medical advice, 0.5% died in the ED, and 5.3% ended "other", which for ED visits includes
admission to the hospital; 66,352 visits (4.8%) have no recorded outcome. By age group, sent
home 96.1%, 95.8%, 91.3%, 86.6%, 78.6%, 71.4% and 61.9% from the youngest to the oldest; "other",
1.3% rising to 18.8%. Nine visits in ten end at home; the share that does not rises steeply with
age, and the "other" category is where admissions hide.

In [ ]:
ended = category_counts(analysis([outcome]).describe(), outcome)
display(ended[["level", "count", "percent", "missing"]].style.format({"percent": PCT}).hide(axis="index"))

by_age = crosstab(age_group, outcome, event="1", event_label="sent home")
print(f"Chi-squared Test, discharge group by age group: chi2 = {by_age.attrs['chi2']:,.0f}, p = {fmt_p(by_age.attrs['p'])}")
levels = list(labels_of(outcome).values())
shares = by_age.set_index("Age group")[levels].div(by_age.set_index("Age group")["n"], axis=0) * 100
display(shares.style.format(PCT))
ax = shares.plot(kind="bar", stacked=True, figsize=(8, 3.8), rot=30,
                 color=["#4C72B0", "#8172B2", "#55A868", "#DD8452", "#C44E52", "#999999"])
ax.set_ylabel("% of the visits of the age group"); ax.set_xlabel(""); ax.set_title("How the visit ended, by age group")
ax.legend(frameon=False, fontsize=7, loc="lower left")
plt.tight_layout()
plt.show()

## Step 6. State the result

A model answer:

> In 1,368,648 sampled US emergency department visits of 2020 to 2022, patients were 42 years old
> on average and 54% were women; one visit in four was coded with a symptom as first diagnosis
> and one in six with an injury. Injuries were most frequent in children and the very old, chest
> pain in middle age, asthma in the young, alcohol-related visits at 45 to 64. Nine visits in ten
> ended with the patient going home, 96% under 18 but 62% at 85 and over. These are unweighted
> sample counts; the NCHS condition flags exist only in the years NCHS delivered them, the curated
> flags in every year.

**Questions for discussion.** The age histogram has a peak in the first years of life and another
in the late twenties: give a clinical reason for each. A visit "without any diagnosis" (3.3%):
what could it be? Why would a national statistics office deliver the 20 most frequent condition
categories of each file rather than all of them? In Step 5 the "other" outcome includes
admission: which of the other percentages would you trust most to compare with your own hospital?

**Optional extension.** Injuries and pregnancy by sex, and the number of diagnoses per visit,
which affects every condition percentage of this scenario.

In [ ]:
by_sex = rates_by(sex, [injury, pregnancy]).set_index("Sex")
display(by_sex.style.format({c: PCT for c in by_sex.columns if c != "n"}))
display(numeric_summary(analysis([n_dx]).describe(), [n_dx]).style.format({"mean": "{:.1f}", "sd": "{:.1f}"}).hide(axis="index"))